# Cup of Coffee — Amedspor match-day reel
### Real image-to-video on a free GPU

Generates a genuine **LTX-Video 2B** image-to-video clip conditioned on the real
Cup of Coffee product photo. Vertical 9:16, 576×1024, 24 fps.

This is real generative video. There is no slideshow, no pan/zoom over a still,
no FFmpeg-synthesised motion and no mock output anywhere in this notebook —
and the verification cell at the end **measures** inter-frame change and fails
if the result is effectively frozen.

---
### Before you run: switch on the GPU
**Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save**

Then **Runtime → Run all**. Everything after that is automatic.

## 1 · Confirm the GPU

In [ ]:
import subprocess, sys

out = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                      '--format=csv,noheader'], capture_output=True, text=True)
if out.returncode != 0:
    sys.exit('No GPU. Runtime -> Change runtime type -> T4 GPU, then Run all again.')

name, mem = [p.strip() for p in out.stdout.strip().split(',')]
gib = int(mem.split()[0]) / 1024
print(f'GPU: {name} — {gib:.1f} GiB')
assert gib >= 10, f'Need >= 10 GiB for ltx-2b-i2v-576p; this GPU has {gib:.1f} GiB.'
print('OK — enough VRAM for the LTX 2B profile.')

## 2 · Install dependencies
_~2 minutes. Torch is already present in Colab._

In [ ]:
%pip install -q --upgrade 'diffusers>=0.35.1' 'transformers>=4.49.0' \
    accelerate safetensors sentencepiece imageio imageio-ffmpeg pillow
print('dependencies installed')

## 3 · Fetch the repository and the real brand assets

In [ ]:
import os, pathlib

if not pathlib.Path('ai-video-studio').exists():
    !git clone --depth 1 https://github.com/alneval20/ai-video-studio.git

os.chdir('/content/ai-video-studio')
assets = pathlib.Path('public')
product = assets / 'cup_of_coffee_HD_preserved.png'
assert product.exists(), 'Product asset missing from the repository.'
print('assets:', *[p.name for p in assets.iterdir() if p.is_file()], sep='\n  ')

## 4 · Prepare the init frame

The source photo is stored **landscape but rotated 90°**, so it is uprighted,
then centre-cropped to exactly 576×1024 — 9:16 and divisible by 32, which LTX
requires. This only conditions the generator; it does not animate anything.

In [ ]:
from PIL import Image, ImageOps

W, H = 576, 1024

img = ImageOps.exif_transpose(Image.open(product)).convert('RGB')
if img.width > img.height:
    img = img.rotate(-90, expand=True)   # upright the sideways original

scale = max(W / img.width, H / img.height)
img = img.resize((round(img.width * scale), round(img.height * scale)), Image.LANCZOS)
left, top = (img.width - W) // 2, (img.height - H) // 2
init_frame = img.crop((left, top, left + W, top + H))

assert init_frame.size == (W, H)
assert W % 32 == 0 and H % 32 == 0, 'LTX requires both dimensions divisible by 32.'
init_frame.save('init_frame.png')
print(f'init frame: {init_frame.size[0]}x{init_frame.size[1]}')
init_frame

## 5 · The prompt

Compiled by the studio pipeline for this exact shot — director → shot planner →
camera / realism / consistency → prompt compiler. Not hand-written here.

In [ ]:
PROMPT = "SCENE: Cinematic commercial footage, vertical format, filmed in the real Cup of Coffee cafe interior: dark charcoal ribbed counter front, warm butcher-block bar top, black industrial pendant lamps, a dense green living plant wall, exposed grey brick and a polished concrete floor at night.\nSUBJECT: A clear ribbed plastic Cup of Coffee cup of iced latte, layered espresso over milk, large clear ice cubes, beaded condensation on the outside is the focus of the frame.\nACTION: Extreme macro inside the iced latte: ice cubes settle and rotate, milk cascades down through the espresso, a condensation droplet runs down the outside of the cup.\nREFERENCE: Reproduce faithfully: upright iced drink and dessert reference; prepared as the I2V init frame before generation. Preserve portioning and layering, garnish placement, surface texture, colour and doneness. Reproduce faithfully: the real cafe counter, lighting and interior identity. Preserve room layout, materials and finishes, light source positions. Reproduce faithfully: drink variety reference only; its weekday/product labels must never be generated into footage. Preserve colour palette, contrast and grade, grain and texture.\nCAMERA: Shot on a gimbal-stabilised rig, giving a slightly compressed, flattering perspective. The shot is an extreme macro frame filling the image with a single surface at eye level with the subject, with the lens sitting just above the table surface.\nREALISM: Photorealistic real-world footage captured on a real camera, not rendered or illustrated. Temporal continuity — every element persists coherently from the first frame to the last. Physically believable lighting: shadows, reflections and falloff agree with the visible light sources."

NEGATIVE = "CGI render, 3D animation, video-game look, illustration, cartoon, AI morphing, objects transforming into other objects, features drifting over time, warped packaging, changing product shape, duplicated products, geometry melting, container morphing"

WIDTH, HEIGHT = 576, 1024
NUM_FRAMES = 73          # 8n+1, required by the LTX temporal VAE
FPS = 24
SEED = 212026
STEPS = 30
GUIDANCE = 5

assert (NUM_FRAMES - 1) % 8 == 0
print(f'{NUM_FRAMES} frames @ {FPS}fps = {NUM_FRAMES/FPS:.2f}s at {WIDTH}x{HEIGHT}')
print()
print(PROMPT)

## 6 · Load LTX-Video 2B
_First run downloads the checkpoint (~2–4 min)._

In [ ]:
import torch
from diffusers import LTXImageToVideoPipeline

pipe = LTXImageToVideoPipeline.from_pretrained(
    'Lightricks/LTX-Video',
    torch_dtype=torch.bfloat16,
)
# Offload rather than .to('cuda'): keeps headroom on a 16 GiB free-tier card.
pipe.enable_model_cpu_offload()
pipe.vae.enable_tiling()
print('pipeline ready on', torch.cuda.get_device_name(0))

## 7 · Generate
_The actual render. ~3–6 minutes on a T4._

In [ ]:
import time

generator = torch.Generator(device='cuda').manual_seed(SEED)
started = time.time()

result = pipe(
    image=init_frame,
    prompt=PROMPT,
    negative_prompt=NEGATIVE,
    width=WIDTH,
    height=HEIGHT,
    num_frames=NUM_FRAMES,
    num_inference_steps=STEPS,
    guidance_scale=GUIDANCE,
    generator=generator,
)

frames_out = result.frames[0]
print(f'generated {len(frames_out)} frames in {(time.time()-started)/60:.1f} min')

## 8 · Export H.264 MP4

In [ ]:
from diffusers.utils import export_to_video
import pathlib

OUT = pathlib.Path('/content/outputs')
OUT.mkdir(parents=True, exist_ok=True)
mp4 = OUT / 'amedspor_ltx_i2v_576x1024.mp4'

export_to_video(frames_out, str(mp4), fps=FPS)
print(f'{mp4}  ({mp4.stat().st_size/1024:.0f} KB)')

## 9 · Verify it is real video

Measures mean absolute change between consecutive frames. A still image,
a slideshow or a frozen generation scores ~0 and **fails here**.

In [ ]:
import numpy as np

arr = np.stack([np.asarray(f, dtype=np.float32) for f in frames_out])
deltas = np.abs(np.diff(arr, axis=0)).mean(axis=(1, 2, 3))
mean_delta = float(deltas.mean())

print(f'frames            {len(frames_out)}')
print(f'mean frame delta  {mean_delta:.3f}  (0 = frozen)')
print(f'min / max         {deltas.min():.3f} / {deltas.max():.3f}')

assert mean_delta > 0.35, (
    f'Effectively frozen (delta {mean_delta:.3f}). This is NOT real temporal video.'
)
print('\nPASS — genuine frame-to-frame motion.')

## 10 · Watch, then download

In [ ]:
from IPython.display import Video, display
display(Video(str(mp4), embed=True, width=360))

In [ ]:
from google.colab import files
files.download(str(mp4))

---
### What to expect

LTX 2B produces believable atmospheric motion — light shifting across the cup,
gentle camera drift, condensation reading as wet. It is a 2B model, so
ice-cube refraction and fine liquid detail are its weakest areas; that is the
documented trade-off of the free path (`docs/FREE-GPU.md`).

If the motion is too subtle, raise `STEPS` to 40 or re-run with a different
`SEED` — each attempt is a couple of minutes and costs nothing.

**Campaign text is never generated into the footage.** `%21` and the Turkish
copy are composited afterwards as clean overlay layers by
`src/lib/compose/amedspor-compositor.ts`.